# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
import os
from pydantic import BaseModel, Field
from tavily import TavilyClient
from dotenv import load_dotenv
from lib.agents import Agent
from lib.llm import LLM
from lib.vector_db import VectorStoreManager, CorpusLoaderService
from lib.tooling import tool

In [11]:
load_dotenv()

assert os.getenv("OPENAI_API_KEY") is not None
assert os.getenv("TAVILY_API_KEY") is not None

assert os.getenv("CHROMA_API_KEY") is not None
assert os.getenv("CHROMA_TENANT") is not None
assert os.getenv("CHROMA_DATABASE") is not None

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

database = VectorStoreManager(OPENAI_API_KEY)
loader = CorpusLoaderService(database)
loader.load_json_data_from_path(
    store_name="udaplay",
    data_dir="games"
    )
game_store = database.get_or_create_store("udaplay")

Items in collection: 213


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
@tool
def retrieve_game(query: str) -> list[str]:
    """
    Tool Docstring:
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry. You'll receive results as list. Each element contains:
        - Platform: like Game Boy, Playstation 5, Xbox 360...)
        - Name: Name of the Game
        - YearOfRelease: Year when that game was released for that platform
        - Description: Additional details about the game
    """
    results = game_store.query(query_texts=[query], n_results=3)
    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]

    if not documents:
        return ["No matching game found in the internal database."]

    return [
        f"Platform: {meta.get('Platform')} | Name: {meta.get('Name')} | "
        f"YearOfRelease: {meta.get('YearOfRelease')} | Publisher: {meta.get('Publisher')} | "
        f"Description: {meta.get('Description')}"
        for meta in metadatas
    ]


#### Evaluate Retrieval Tool

In [ ]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the retrieved documents are enough to answer the question")
    description: str = Field(description="Detailed explanation supporting the verdict")

@tool
def evaluate_retrieval(question: str, retrieved_docs: list[str]):
    """
    Tool Docstring:
       Based on the user's question and on the list of retrieved documents,
       it will analyze the usability of the documents to respond to that question.
       args:
       - question: original question from user
       - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
       The result includes:
       - useful: whether the documents are useful to answer the question
       - description: description about the evaluation result
    """
    judge = LLM(model="gpt-4o-mini", temperature=0.0)

    docs_block = "\n".join(f"- {doc}" for doc in retrieved_docs) if retrieved_docs else "(no documents retrieved)"
    prompt = (
        f"QUESTION:\n{question}\n\n"
        f"RETRIEVED DOCUMENTS:\n{docs_block}\n\n"
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not."
    )
    response = judge.invoke(prompt, response_format=EvaluationReport)
    report = EvaluationReport.model_validate_json(response.content)
    return report.model_dump()

#### Game Web Search Tool

In [ ]:
@tool
def game_web_search(question: str):
    """
    Tool Docstring:
       Searches the web for information about the video game industry when the
       internal database doesn't have a good enough answer.
       args:
       - question: a question about the game industry.
    Returns a list of strings, each with a title, a short snippet and the source URL.
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(question, max_results=3)
    results = response.get("results", [])

    if not results:
        return ["No web results found."]

    return [
        f"{r.get('title')}: {r.get('content')} (Source: {r.get('url')})"
        for r in results
    ]

### Agent

In [ ]:
agent = Agent(
    model_name="gpt-4o-mini",
    instructions=(
        "You are UdaPlay's Research Agent, an expert on the video game industry.\n\n"
        "For every user question, follow this decision process:\n"
        "1. Call `retrieve_game` with a focused search query to look up the internal vector database first.\n"
        "2. Call `evaluate_retrieval` passing the original question and the documents you got back, "
        "to judge whether they are sufficient to answer confidently.\n"
        "3. If `evaluate_retrieval` says the documents ARE useful, answer using ONLY that internal "
        "information and cite it as '(Source: internal database)'.\n"
        "4. If `evaluate_retrieval` says the documents are NOT useful, call `game_web_search` with the "
        "question (rephrase it if that helps) and answer using those results, citing the source URL(s).\n\n"
        "Never skip steps 1 and 2. Be transparent about which source you used, and end every final "
        "answer with a short 'Sources:' line."
    ),
    tools=[retrieve_game, evaluate_retrieval, game_web_search]
)

In [16]:
example_queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

session_id = "udaplay-demo"

for query in example_queries:
    print(f"\n{'='*80}\nUser: {query}\n{'='*80}")
    run = agent.invoke(query, session_id=session_id)
    final_state = run.get_final_state()

    for msg in final_state["messages"]:
        if msg.role == "assistant" and msg.tool_calls:
            calls = ", ".join(f"{c.function.name}({c.function.arguments})" for c in msg.tool_calls)
            print(f"[reasoning] Agent decided to call: {calls}")
        elif msg.role == "tool":
            print(f"[tool result - {msg.name}] {msg.content[:300]}")

    final_answer = final_state["messages"][-1].content
    print(f"\nFinal Answer:\n{final_answer}")


User: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[reasoning] Agent decided to call: retrieve_game({"query":"Pokémon Gold and Silver release date"})
[tool result - retrieve_game] "['Platform: Game Boy Color | Name: Pok\u00e9mon Gold and Silver | YearOfRelease: 1999 | Publisher: Nintendo | Description: Second-generation Pok\u00e9mon games introducing new regions, Pok\u00e9mon, and gameplay mechanics.', 'Platform: GB | Name: Pokemon Gold/Pokemon Silver | YearOfRelease: 1999 | 
[reasoning] Agent decided to call: evaluate_retrieval({"question":"When was Pokémon Gold and Silver released?","retrieved_docs":["Platform: Game Boy Color | Name: Pokémon G

### (Optional) Advanced

In [17]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes